# Phase 4: the identity, not a threshold

The ratio $d$ obeys $\Delta d_t = d_{t-1}\frac{r_t-g_t}{1+g_t} - pb_t + sf_t$ (snowball, primary, stock-flow residual).
Setting $\Delta d = 0$ gives the debt-stabilizing primary balance $pb^*$ and the debt-stabilizing growth rate
$g^* = (d\,r - pb)/(d + pb)$. A path is in the doom loop when $pb^*$ exceeds what is feasible for N consecutive
periods. No debt level is chosen. Derivations: `docs/math.md` 1-4; decision `docs/decisions/0006`.

$r$ is the bottom-up effective rate from Phase 2. Second differences are decomposed to say what is accelerating the
ratio; third differences are tested as a leading indicator only.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
import bond_sim.notebook as nb
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
plt.rcParams["figure.dpi"] = 110

# ── parameters: change and rerun ─────────────────────────────────────────────
AS_OF = pd.Timestamp("2026-09-15")     # every loader filters on this date (point-in-time contract)
K = 1000                                 # Monte Carlo paths (raise for the paper)
FORCE_REFRESH = False                  # True re-downloads every series and rebuilds cached steps
os.environ["BOND_SIM_CACHE"] = "0" if FORCE_REFRESH else "1"
import bond_sim.config as bcfg
CFG_HASH = bcfg.load().content_hash()   # every cached step is keyed by the configuration that produced it
ctx = nb.cached(f"ctx_{AS_OF.date()}_{CFG_HASH}", lambda: nb.load_context(AS_OF), refresh=FORCE_REFRESH)
print(ctx.grid, "| config hash", CFG_HASH, "| series:", ctx.w.shape[1])

In [ ]:
book, auctions, agg, short = nb.cached(f"book_{AS_OF.date()}_{CFG_HASH}", lambda: nb.build_book(ctx), refresh=FORCE_REFRESH)
from bond_sim.sim import decompose, stabilizing_growth, stabilizing_primary_balance, FeasiblePB, evaluate_history, debt_limit
from bond_sim.sim.premium import LinearPremium, ThresholdPremium, NoPremium, premium_from_config
h = nb.history_frame(ctx, agg)
dec = decompose(h["d"], h["r_eff"], h["g_nom"], h["pb"])
fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
(100 * dec[["snowball", "primary", "residual"]]).plot.bar(ax=ax[0], stacked=True, width=1.0, linewidth=0)
ax[0].plot(range(len(dec)), 100 * dec["delta_d"], color="k", lw=1.2, label="delta d"); ax[0].legend(fontsize=8)
ax[0].set_title("Quarterly change in debt/GDP (pct pts): snowball vs primary vs stock-flow residual"); ax[0].set_xticks([])
(100 * dec["r_minus_g"]).plot(ax=ax[1]); ax[1].axhline(0, color="k", lw=0.8); ax[1].set_title("r minus g (pct pts, annualized)"); ax[1].set_xlabel("")
plt.tight_layout(); plt.show()

## 4.1 What would it take: $pb^*$ and $g^*$ against what actually happened

In [ ]:
d_lag = h["d"].shift(1)
h["pb_star"] = stabilizing_primary_balance(d_lag, h["r_eff"], h["g_nom"])
h["g_star"] = stabilizing_growth(d_lag, h["r_eff"], h["pb"])
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
(100 * h[["pb", "pb_star"]]).plot(ax=ax[0]); ax[0].set_title("Primary balance: actual vs debt-stabilizing (% GDP)"); ax[0].set_xlabel("")
(100 * h[["g_nom", "g_star"]]).rolling(4).mean().plot(ax=ax[1]); ax[1].set_title("Nominal growth: actual vs debt-stabilizing (4q avg, %)"); ax[1].set_xlabel("")
plt.tight_layout(); plt.show()
print(f"latest: pb {100*h.pb.iloc[-1]:.2f}%  pb* {100*h.pb_star.iloc[-1]:.2f}%  |  g {100*h.g_nom.iloc[-4:].mean():.2f}%  g* {100*h.g_star.iloc[-1]:.2f}%  |  r_eff {100*h.r_eff.iloc[-1]:.2f}%  d {100*h.d.iloc[-1]:.0f}%")

## 4.2 What is feasible

Two benchmarks from the data: the historical envelope (maximum observed primary balance) and a fiscal reaction function
with fatigue (cubic in lagged debt, unemployment as the cyclical control). A negative cubic term means the response
weakens at high debt. With a premium that makes $r$ increase in $d$, the reaction function implies a debt limit.

In [ ]:
feas = FeasiblePB.from_history(h["pb"], h["d"], h["u"], quantile=ctx.cfg.doomloop.feasible_quantile)
fr = feas.reaction
print(f"envelope (max primary balance): {100*feas.envelope:.2f}% of GDP")
print(f"reaction function: pb = {fr.coef[0]:.4f} + {fr.coef[1]:.4f} d + {fr.coef[2]:.4f} d^2 + {fr.coef[3]:.4f} d^3 + {fr.coef[4]:.4f} (u - mean)   R2 {fr.r2:.2f}, n {fr.n}, fatigue: {fr.fatigue}")
grid_d = np.linspace(0.3, 2.5, 100)
r_now, g_now = float(h.r_eff.iloc[-1]), float(h.g_nom.iloc[-8:].mean())
prem_models = [("no premium", NoPremium()), ("linear premium (P-01 placeholder)", premium_from_config(ctx.cfg.doomloop.risk_premium)),
               ("threshold premium (P-02 placeholder)", ThresholdPremium(ctx.cfg.doomloop.risk_premium.bps_per_pct_debt_gdp, ctx.cfg.doomloop.risk_premium.anchor_debt_gdp_pct,
                                                                         ctx.cfg.doomloop.risk_premium.threshold_debt_gdp_pct, ctx.cfg.doomloop.risk_premium.threshold_extra_bps_per_pct))]
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(100 * grid_d, 100 * feas.evaluate(grid_d, which="reaction"), label="feasible pb (reaction function)")
ax.axhline(100 * feas.envelope, color="grey", ls="--", label=f"feasible pb (envelope, q{ctx.cfg.doomloop.feasible_quantile})")
for name, prem in prem_models:
    r_of_d = lambda x, p=prem: r_now + float(p.premium(np.array([100 * x]))[0]) / 100.0
    pbs = 100 * stabilizing_primary_balance(grid_d, np.array([r_of_d(x) for x in grid_d]), np.full_like(grid_d, g_now))
    ax.plot(100 * grid_d, pbs, label=f"pb* needed, {name}")
ax.set_xlabel("debt / GDP (%)"); ax.set_ylabel("% of GDP"); ax.set_ylim(-6, 8); ax.legend(fontsize=8)
ax.set_title(f"Required vs feasible primary balance at today's r_eff {100*r_now:.2f}% and recent g {100*g_now:.2f}%"); plt.show()

**The debt limit depends on r minus g, not on debt alone.** With r below g (today: effective rate 3.3% against
nominal growth above 5%), every debt level is stabilizable by a modest deficit and the reaction function implies no
meaningful limit. The limit only appears once the base rate exceeds growth, so it is reported as a function of an
assumed r minus g spread (0, +1, +2 points on top of today's growth) and of the premium model, against both feasibility
benchmarks. That table is the paper's answer to "how much room is there", conditional on the one number nobody knows.

In [ ]:
rows = []
for rg in (0.0, 0.01, 0.02):
    for name, prem in prem_models:
        r_of_d = lambda x, p=prem, rg=rg: g_now + rg + float(p.premium(np.array([100 * x]))[0]) / 100.0
        for which in ("actual", "envelope", "reaction"):
            feas_w = FeasiblePB(envelope=feas.envelope, reaction=feas.reaction if which == "reaction" else None)
            if which == "envelope":
                pbs = stabilizing_primary_balance(grid_d, np.array([r_of_d(x) for x in grid_d]), np.full_like(grid_d, g_now))
                over = np.nonzero(pbs > feas.envelope)[0]
                lim = float(grid_d[over[0]]) if len(over) else None
            else:
                lim = debt_limit(feas_w, r_of_d, g_now)
            rows.append({"r minus g (pct pts)": 100 * rg, "premium": name, "benchmark": which,
                         "debt limit (% GDP)": None if lim is None else round(100 * lim)})
display(pd.DataFrame(rows).pivot(index=["r minus g (pct pts)", "premium"], columns="benchmark", values="debt limit (% GDP)"))
print(f"today: debt/GDP {100*h.d.iloc[-1]:.0f}%, r_eff {100*r_now:.2f}%, recent nominal growth {100*g_now:.2f}% (r minus g = {100*(r_now-g_now):+.2f} pts)")

## 4.3 The trigger on history

Which quarters since 1980 would have flagged, under each persistence window and benchmark? A definition that flags
nothing is useless; one that flags every quarter is worse. This is the calibration exercise for N and the quantile,
the only two free numbers.

In [ ]:
rows = []
for quantile in (1.0, 0.9, 0.75):
    feas_q = FeasiblePB.from_history(h["pb"], h["d"], h["u"], quantile=quantile, fit_reaction=False)
    for require in (True, False):
        for n in (2, 4, 8):
            H = evaluate_history(h, feas_q, n_quarters=n, which="envelope", require_r_gt_g=require)
            flagged = H.index[H["triggered"].fillna(False)]
            rows.append({"envelope quantile": quantile, "feasible pb %GDP": round(100 * feas_q.envelope, 2), "require r>g": require, "N quarters": n,
                         "breach quarters": int(H["breach"].sum()), "triggered quarters": len(flagged),
                         "first": flagged.min().date() if len(flagged) else None, "last": flagged.max().date() if len(flagged) else None})
for require in (True, False):
    H = evaluate_history(h, feas, n_quarters=4, which="reaction", require_r_gt_g=require)
    flagged = H.index[H["triggered"].fillna(False)]
    rows.append({"envelope quantile": "reaction fn", "feasible pb %GDP": None, "require r>g": require, "N quarters": 4,
                 "breach quarters": int(H["breach"].sum()), "triggered quarters": len(flagged),
                 "first": flagged.min().date() if len(flagged) else None, "last": flagged.max().date() if len(flagged) else None})
display(pd.DataFrame(rows))
H = evaluate_history(h, feas, n_quarters=4, which=ctx.cfg.doomloop.feasible_benchmark, require_r_gt_g=ctx.cfg.doomloop.trigger_require_r_gt_g)
fig, ax = plt.subplots(figsize=(13, 4))
(100 * H["gap"]).plot(ax=ax, label="pb* minus feasible pb (pct pts of GDP)"); ax.axhline(0, color="k", lw=0.8)
for d in H.index[H["triggered"].fillna(False)]:
    ax.axvspan(d, d + pd.DateOffset(months=3), color="red", alpha=0.15)
ax.set_title("Sustainability gap on history (red: triggered under N=4)"); ax.legend(); ax.set_xlabel(""); plt.show()

## 4.4 Second and third differences

Acceleration decomposed into the snowball and the primary balance, and the third difference as a candidate early warning:
how often was it positive in the two years before a breach began, versus in general?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
(100 * dec[["delta2_snowball", "delta2_primary"]]).rolling(4).mean().plot(ax=ax[0]); ax[0].axhline(0, color="k", lw=0.8)
ax[0].set_title("Acceleration of debt/GDP: snowball vs primary (4q avg)"); ax[0].set_xlabel("")
(100 * dec["delta3_d"]).rolling(4).mean().plot(ax=ax[1]); ax[1].axhline(0, color="k", lw=0.8); ax[1].set_title("Third difference (4q avg)"); ax[1].set_xlabel("")
plt.tight_layout(); plt.show()
breach_start = H["breach"] & ~H["breach"].shift(1, fill_value=False)
pre = pd.Series(False, index=H.index)
for d in H.index[breach_start]:
    pre.loc[(H.index < d) & (H.index >= d - pd.DateOffset(years=2))] = True
d3 = dec["delta3_d"].reindex(H.index)
print(f"share of quarters with positive third difference: overall {float((d3 > 0).mean()):.2f}, in the 2 years before a breach begins {float((d3[pre] > 0).mean()):.2f}  (n pre-breach quarters: {int(pre.sum())})")